<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_hotelchain_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *HotelChain West* dans Power BI Desktop. Il est conçu pour toi si :

- tu as déjà manipulé Excel et tu sais ce qu'est une formule,
- tu as installé Power BI Desktop sur ta machine,
- tu comprends ce qu'est une jointure entre deux tables.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |
| 🚀 | L'optimisation ou la variante avancée |

### Le contexte métier

**HotelChain West** est une chaîne hôtelière ouest-africaine de 5 établissements (Douala 3★, Cocody 3★, Dakar 4★, Plateau 4★, Accra 5★) avec une stratégie de pilotage data-driven. Quatre métriques industrielles structurent le pilotage :

1. **RevPAR** (Revenue Per Available Room) — la mesure de référence du secteur hôtelier (= ADR × Taux d'occupation)
2. **ADR** (Average Daily Rate) — prix moyen vendu par nuit
3. **Taux d'occupation** — nuits vendues / capacité théorique
4. **Ratio extras / chambres** — part des services annexes (restaurant, spa, conférence) dans le revenu total. Cible 15 %, benchmark hôtellerie haut de gamme

Le dashboard que tu vas construire répond à 5 questions stratégiques :

| Page | Question |
|---|---|
| 1 — Vue executive | Quelle est la santé globale de la chaîne ce trimestre ? |
| 2 — Occupation & Saisonnalité | Quel hôtel performe, à quelle saison, avec quel pic / creux ? |
| 3 — Revenus & Tarification | À quel prix vendons-nous, quel hôtel a le meilleur sweet spot ADR × Occupation ? |
| 4 — Clients & Satisfaction | Qui sont nos clients, quelle est leur satisfaction, quelle est la valeur des fidèles ? |
| 5 — Extras | Combien rapportent les services annexes, où est le potentiel d'upsell ? |

## 🗺️ Sommaire détaillé

| Partie | Section | Durée |
|---|---|---|
| **I — Fondations** | Sources · Import CSV · Auto Date/Time | 30 min |
| **II — Modélisation** | Étoile · Calendrier · 9 relations · Marquage | 50 min |
| **III — Table `_Mesures`** | Création | 10 min |
| **IV — 44 mesures DAX** | 18 dossiers numérotés (KPIs, Évolution, Cartes hôtel, Heatmaps HTML, Recommandations) | 3 h |
| **V — Design** | Charte Dark Luxury · Mockup PPTX → PNG · Visuel HTML Content | 40 min |
| **VI — 6 pages** | Cover + Vue executive / Occupation / Revenus / Clients / Extras | 1 h 30 |
| **VII — Finitions** | Slicers · Navigation · Sweet spot scatter | 25 min |
| **VIII — Validation** | Checklist · Pièges · Storytelling · Annexes | 45 min |

**Total** ≈ **7 h 30** de travail effectif.

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

### 📘 Concept clé — le « grain »

Une ligne de chaque table représente une entité bien identifiée. Avant d'écrire la moindre formule, complète mentalement : *« Une ligne de cette table = un(e) ____ »*.

### Inventaire des 9 sources

| Fichier | Grain | Rôle | Volumétrie |
|---|---|---|---|
| `hotels.csv` | 1 hôtel de la chaîne (5 hôtels) | Dimension | 5 lignes |
| `chambres.csv` | 1 chambre × 1 hôtel (avec son type et tarif catalogue) | Dimension | ~700 lignes |
| `clients.csv` | 1 client unique (avec nationalité et fidélité) | Dimension | ~2 700 lignes |
| `reservations.csv` | 1 réservation × 1 chambre × 1 client | Fait principal | ~8 000 lignes |
| `services.csv` | 1 prestation extra (restaurant, spa, transport...) liée à une réservation | Fait | ~5 500 lignes |
| `paiements.csv` | 1 transaction de paiement | Fait | ~9 500 lignes |
| `hotelchain_occupation.csv` | 1 hôtel × 1 mois avec nuits vendues / capacité (output EDA) | Fait dérivé | 60 lignes |
| `hotelchain_revpar.csv` | 1 hôtel avec ADR et RevPAR (output EDA) | Fait dérivé | 5 lignes |
| `hotelchain_clients_clv.csv` | 1 client avec son CLV et son segment quartile (output ML) | Fait dérivé | ~2 700 lignes |
| `hotelchain_canaux.csv` | 1 canal de réservation × revenu agrégé (output EDA) | Fait dérivé | 5 lignes |

### ⚠️ Piège fréquent — `taux_annulation_pct` mal inféré

La colonne `taux_annulation_pct` arrive parfois en **DateTime** au lieu de Decimal (Power Query interprète `2.8` comme une date). Solution : forcer le type **Nombre décimal** dans Power Query, ou contourner avec `DAY([col]) + MONTH([col])/10` dans le DAX (hack documenté dans la mesure `Taux Annulation`).

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit, aucune ré-installation.

L'inconvénient : il faut une connexion internet au premier chargement. Une fois publié sur le service Power BI, le rapport peut être planifié pour rafraîchir tout seul.

### Les 10 URLs à utiliser

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/data/hotels.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/data/chambres.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/data/reservations.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/data/services.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/corrige/outputs/hotelchain_occupation.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/corrige/outputs/hotelchain_revpar.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/corrige/outputs/hotelchain_clients_clv.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/corrige/outputs/hotelchain_canaux.csv
```

> 💡 *Remplace par ton chemin de dépôt réel. Si les CSV sont en local pour l'instant, utilise `Obtenir les données → Texte/CSV` ; le reste de la procédure est identique.*


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/01_powerquery_10_requetes.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**.

Sans ça, Power BI crée une LocalDateTable cachée pour chaque colonne de date — sur ce projet (`reservations[date_arrivee]`, `reservations[date_depart]`, `paiements[date_paiement]`, `services[date_service]`...), c'est 4-5 tables fantômes en moins.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_options_auto_datetime.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>


---
# II — Modéliser les données

## 2.1 Le schéma en étoile

### 📘 Concept clé

Schéma en étoile : faits au centre, dimensions autour. Relations 1→N uniquement, OneDirection. Sur HotelChain on a une particularité : **5 tables "output ML/EDA"** (occupation, revpar, clv, canaux) qui se branchent en parallèle des tables sources. C'est volontaire : on évite de re-calculer dans Power BI ce que Python a déjà calculé.

### Diagramme du modèle HotelChain West

```
                      +------------------+
                      |   Calendrier     |
                      +--------+---------+
                               | 1
                               | N
                  +------------+-------------+
                  |  hotelchain_occupation   |
                  +------------+-------------+
                               | N
                               | 1
    +------------+   1    N    +-------+   N    1   +------------+
    |   hotels   |--------------- nom -+----------- |  chambres  |
    +-----+------+                                  +------+-----+
          |1                                              |1
          | N                                             | N
    +-----+------+   1    N    +-------------+   N    1   +------------------+
    |  services  |<-------------+ reservations+----------- | hotelchain_clients_clv |
    +------------+              +-----+-------+            +------------------+
                                      | N
                                      | 1
                              +-------+----------+
                              | hotelchain_canaux|
                              +------------------+

                      +-------------------+
                      | hotelchain_revpar |  (1 ligne par hôtel — relié à hotels[nom])
                      +-------------------+
```

## 2.2 Créer la table Calendrier

**Modélisation → Nouvelle table** :

```dax
Calendrier = 
VAR _start = DATE(2022,1,1)
VAR _end   = DATE(2024,12,31)
RETURN
ADDCOLUMNS(
    CALENDAR(_start, _end),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Mois_Lettre",   UPPER(LEFT(FORMAT([Date], "mmm", "fr-FR"), 1)),
    "Annee_Mois",    FORMAT([Date], "yyyy-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Saison",        SWITCH(TRUE(),
                       MONTH([Date]) IN {12,1,2}, "Hiver",
                       MONTH([Date]) IN {3,4,5}, "Printemps",
                       MONTH([Date]) IN {6,7,8}, "Ete",
                       "Automne")
)
```

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/03_calendrier_dax.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 2.3 Établir les 9 relations

| # | De (1) | Clé | Vers (N) | Clé | Direction |
|---|---|---|---|---|---|
| 1 | `Calendrier` | `Date` | `hotelchain_occupation` | `date` | Single |
| 2 | `hotels` | `nom` | `hotelchain_occupation` | `hotel` | Single |
| 3 | `hotels` | `nom` | `hotelchain_revpar` | `hotel` | Single |
| 4 | `hotels` | `hotel_id` | `chambres` | `hotel_id` | Single |
| 5 | `hotels` | `hotel_id` | `services` | `hotel_id` | Single |
| 6 | `chambres` | `chambre_id` | `reservations` | `chambre_id` | Single |
| 7 | `hotelchain_clients_clv` | `client_id` | `reservations` | `client_id` | Single |
| 8 | `hotelchain_canaux` | `canal` | `reservations` | `canal` | Single |
| 9 | `services` | `reservation_id` | `reservations` | `reservation_id` | **Inactive** (relation 2 entre services et reservations, à laisser inactive ou supprimer pour éviter les ambiguïtés) |


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/02_modele_etoile.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 2.4 Marquer Calendrier comme table de dates

Vue Données → `Calendrier` → **Outils de table → Marquer comme table de dates → colonne Date**. Sans ça, `PREVIOUSMONTH` retourne blank silencieusement et la mesure `Variation Revenu %` est cassée.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/marquer_table_date.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, supprime la colonne fictive.

---
# IV — Construire les 44 mesures DAX

### Vue d'ensemble des 18 dossiers

Le projet HotelChain a une organisation très fine (18 dossiers numérotés) qui sépare les fondations (1-4), les enrichissements (5-7), les visuels HTML custom (8-17) et les recommandations dynamiques (18).

| # | Dossier | Mesures | Rôle |
|---|---|---|---|
| 01 | KPIs de base | 6 | Réservations, Revenu, CSAT, Annulation, Total hôtels, Ticket conférence |
| 02 | KPIs opérationnels | 4 | Nuits vendues, Taux occupation, ADR, RevPAR |
| 03 | Évolution | 2 | Revenu mois précédent, Variation % |
| 04 | Services & extras | 4 | Revenu Extras, Total consolidé, Pct Extras, Ratio vs Chambres |
| 05 | Dynamiques | 2 | Sous-titre Page 1, Visuel Breakdown |
| 06 | KPIs Occupation Extrêmes | 6 | Pic / Creux / Moyenne avec leurs labels |
| 07 | Cartes Hôtel | 4 | Hotel Label Étoiles, RevPAR formaté, Insight contextuel |
| 08 | Heatmap Occupation | 2 | Mesure HTML + Titre dynamique |
| 09 | RevPAR Cards | 1 | Bloc HTML 5 cards RevPAR |
| 10 | ADR par Type Chambre | 1 | Bloc HTML par type avec tarif catalogue |
| 12 | Segmentation Clients | 4 | Compteurs Nouveaux / Fidèles + % |
| 13 | CSAT par Hôtel | 1 | Bar chart HTML CSAT vs cible 4.0 |
| 14 | Top Nationalités | 1 | Bar chart HTML Top 5 |
| 15 | CLV Clients Table | 1 | Tableau HTML segmentation par quartile |
| 16 | Services par Catégorie | 1 | Bar chart HTML revenu × ticket moyen |
| 17 | Ratio Extras Cible | 1 | HTML jauge avec cible 15 % |
| 18 | Recommandations Actions | 3 | Textes dynamiques pour les 3 cards d'action |

*Note : il n'y a pas de dossier 11. La numérotation a un trou volontaire.*

## 4.1 Dossier `01. KPIs de base` (6 mesures)

```dax
Total Reservations = COUNTROWS(hotelchain_occupation)
```

```dax
Revenu Total = SUM(hotelchain_canaux[revenu_total_fcfa])
```
*Format : `#,0" FCFA"`. Source = `hotelchain_canaux` (output EDA), pas la table reservations brute, car l'agrégation par canal est déjà faite.*

```dax
CSAT Moyen = AVERAGE(hotelchain_clients_clv[csat_moy])
```
*Format : `0.00`. Cible : ≥ 4.0.*

```dax
Taux Annulation = 
VAR _val = AVERAGE(hotelchain_occupation[taux_annulation_pct])
RETURN _val
```

### ⚠️ Piège — `Taux Annulation`

Si Power Query a inféré la colonne en DateTime (cas observé : `2.8` interprété comme `02/08/2026`), tu as 2 options :
1. **Propre** : Power Query → onglet Transformer → forcer le type **Nombre décimal**
2. **Hack DAX** si tu n'as pas la main : `DAY([col]) + MONTH([col])/10` qui reconstitue le pourcentage depuis la date corrompue

```dax
Total hotel = DISTINCTCOUNT(hotels[hotel_id])
```

```dax
Ticket Moyen Conference = 
CALCULATE(
    AVERAGE(services[montant]),
    services[categorie] = "Salle conference"
)
```
*Indicateur clé du potentiel B2B / événementiel. Valeur attendue ~132 504 FCFA.*

## 4.2 Dossier `02. KPIs opérationnels` (4 mesures)

```dax
Nuits Vendues = SUM(hotelchain_occupation[nuits_vendues])
```

```dax
Taux Occupation = 
DIVIDE(
    SUM(hotelchain_occupation[nuits_vendues]),
    SUM(hotelchain_occupation[capacite_theorique])
)
```

### 📘 Pourquoi ne pas utiliser la colonne `taux_occupation` brute ?

Une moyenne de ratios n'est pas la même chose qu'un ratio de sommes. Si Douala fait 80 % d'occupation sur 100 chambres et Cocody 40 % sur 200 chambres, la *moyenne arithmétique* donne 60 % — alors que le *vrai* taux global est 53 % (160 nuits / 300 capacité). Toujours sommer numérateur et dénominateur séparément.

```dax
ADR = AVERAGE(hotelchain_revpar[adr_fcfa])
```
*Average Daily Rate. À croiser avec Taux Occupation pour identifier le sweet spot tarifaire.*

```dax
RevPAR = AVERAGE(hotelchain_revpar[revpar_fcfa])
```
*Revenue Per Available Room = ADR × Taux Occupation. Mesure de référence de l'industrie hôtelière.*

## 4.3 Dossier `03. Évolution` (2 mesures)

```dax
Revenu Mois Prec = 
CALCULATE(
    [Revenu Total],
    PREVIOUSMONTH(Calendrier[Date])
)
```

```dax
Variation Revenu % = 
DIVIDE([Revenu Total] - [Revenu Mois Prec], [Revenu Mois Prec])
```
*Format : `+0.0%;-0.0%;0%`. Positif = croissance (vert), négatif = baisse (rouge). BLANK sur le premier mois de la période.*

## 4.4 Dossier `04. Services & extras` (4 mesures)

```dax
Revenu Extras = SUM(services[montant])
```

```dax
Revenu Total Consolide = [Revenu Total] + [Revenu Extras]
```

```dax
Pct Extras = DIVIDE([Revenu Extras], [Revenu Total Consolide])
```
*Format : `0.0%`. Part des extras dans le revenu total consolidé. Benchmark industrie hôtelière haut de gamme : 15-25 %.*

```dax
Ratio Extras vs Chambres = DIVIDE([Revenu Extras], [Revenu Total])
```
*Format : `0.0%`. Mesure l'efficacité du cross-sell **par rapport au cœur de métier hébergement** (pas par rapport au consolidé). Valeur attendue ~9.7 %. Benchmark : 15-25 % haut de gamme, 5-10 % standard.*

## 4.5 Dossier `05. Dynamiques` (2 mesures)

```dax
Sous Titre Page 1 = 
VAR _hotel  = SELECTEDVALUE(hotels[nom], "Tous les hôtels")
VAR _annee  = SELECTEDVALUE(Calendrier[Annee], "Toutes années")
VAR _saison = SELECTEDVALUE(Calendrier[Saison], "Toutes saisons")
RETURN _hotel & "  ·  " & _annee & "  ·  " & _saison
```

```dax
Visuel Breakdown Revenu = 
// HTML retournant un breakdown en barres horizontales : Chambres / Extras / Consolidé
// Construit avec le pattern CONCATENATEX + style inline (cf. mesures HTML §4.8)
```

## 4.6 Dossier `06. KPIs Occupation Extrêmes` (6 mesures)

### 📘 Concept clé — pic et creux par hôtel × mois

Sur la page Occupation, on veut afficher *« Douala Janvier 2024 = 7,5 % (pic) »* et *« Accra Juin 2024 = 1,2 % (creux) »*. Ces mesures combinent la valeur extrême et le label texte associé.

```dax
Pic Occupation = 
VAR _tbl = 
    SUMMARIZE(
        hotelchain_occupation,
        hotels[nom], Calendrier[Annee_Mois],
        "@tx", DIVIDE(SUM(hotelchain_occupation[nuits_vendues]), SUM(hotelchain_occupation[capacite_theorique]))
    )
RETURN MAXX(_tbl, [@tx])
```

```dax
Pic Occupation Label = 
VAR _max = [Pic Occupation]
VAR _tbl = SUMMARIZE(hotelchain_occupation, hotels[nom], Calendrier[Annee_Mois], "@tx", [Taux Occupation])
VAR _row = TOPN(1, FILTER(_tbl, [@tx] = _max), [@tx], DESC)
VAR _hotel = SUBSTITUTE(MAXX(_row, hotels[nom]), "HotelChain ", "")
VAR _periode = MAXX(_row, Calendrier[Annee_Mois])
RETURN _hotel & " " & _periode
```
*Renvoie ex: "Douala Janvier 2024". Le `SUBSTITUTE` retire le préfixe `HotelChain` pour l'affichage.*

```dax
Creux Occupation = 
VAR _tbl = SUMMARIZE(hotelchain_occupation, hotels[nom], Calendrier[Annee_Mois], "@tx", [Taux Occupation])
RETURN MINX(_tbl, [@tx])
```

```dax
Creux Occupation Label = 
VAR _min = [Creux Occupation]
VAR _tbl = SUMMARIZE(hotelchain_occupation, hotels[nom], Calendrier[Annee_Mois], "@tx", [Taux Occupation])
VAR _row = TOPN(1, FILTER(_tbl, [@tx] = _min), [@tx], ASC)
VAR _hotel = SUBSTITUTE(MAXX(_row, hotels[nom]), "HotelChain ", "")
VAR _periode = MAXX(_row, Calendrier[Annee_Mois])
RETURN _hotel & " " & _periode
```

```dax
Moyenne Occupation = [Taux Occupation]
```

```dax
Moyenne Occupation Label = 
VAR _h = SELECTEDVALUE(hotels[nom], "Tous les hôtels")
RETURN SUBSTITUTE(_h, "HotelChain ", "")
```

## 4.7 Dossier `07. Cartes Hôtel` (4 mesures)

```dax
Hotel Label Etoiles = 
VAR _h = SELECTEDVALUE(hotels[nom])
VAR _e = SELECTEDVALUE(hotels[etoiles])
RETURN SUBSTITUTE(_h, "HotelChain ", "") & " " & _e & "\u2605"
```
*Affichage : "Douala 3★". Utilisée dans les cartes de la page Occupation (Bilan par hôtel).*

```dax
RevPAR Hotel Format = 
VAR _v = [RevPAR]
RETURN FORMAT(_v, "#,0") & " FCFA"
```

```dax
Hotel Insight = 
VAR _rang = RANKX(ALL(hotels[nom]), [RevPAR], , DESC)
VAR _adr_h = [ADR]
VAR _adr_max = MAXX(ALL(hotels[nom]), [ADR])
VAR _total_h = COUNTROWS(ALL(hotels[nom]))
RETURN SWITCH(TRUE(),
    _rang = 1,           "Leader RevPAR",
    _rang = _total_h,    "Sous-perf.",
    _adr_h = _adr_max,   "ADR eleve",
    _rang = 2,           "2e rang",
    _rang = 3,           "3e rang",
    ""
)
```
*Logique en cascade : Rang 1 → Leader, dernier → Sous-perf., autres → ordinal.*

```dax
Hotel RevPAR FCFA = FORMAT([RevPAR], "#,0") & " FCFA"
```

## 4.8 Dossiers `08-17` — Visuels HTML Content (10 mesures)

### 📘 Concept clé — pourquoi du HTML Content sur ce projet ?

HotelChain pousse loin l'esthétique "Dark Luxury" (fond noir, accents or, gradients). Les visuels natifs Power BI ne savent pas faire :
- une **heatmap 5×12 avec palette dorée 5 paliers** (la matrice native ne gère pas les gradients custom propres),
- des **cards RevPAR avec barres proportionnelles au leader** (100 % = leader, autres pourcentages),
- des **tableaux CLV avec quartiles inversés et badges colorés**.

Pour ces 8 visuels critiques, on utilise le visuel marketplace **HTML Content** (Daniel Marsh-Patrick) qui rend une string DAX comme du HTML. **Pour les visuels plus simples (CSAT par hôtel, Top Nationalités), tu peux utiliser une barre native + mesure couleur** (cf. §4.10).

### Liste des 10 mesures HTML

| Dossier | Mesure | Page | Rôle |
|---|---|---|---|
| 08 | `Heatmap Occupation HTML` | Occupation | Heatmap 5 hôtels × 12 mois, palette or 5 paliers |
| 08 | `Titre Heatmap Occupation` | Occupation | Titre dynamique "Taux d'occupation 2024 — hôtel × mois" |
| 09 | `Cards RevPAR Hotels` | Vue executive | 5 cards verticales avec barre proportionnelle au leader |
| 10 | `ADR par Type Chambre HTML` | Revenus | 4 lignes avec barre + ADR k FCFA + nb réservations |
| 13 | `CSAT par Hotel HTML` | Clients | Bar chart avec ligne cible pointillée à 4.0 |
| 14 | `Top Nationalites HTML` | Clients | Top 5 nationalités (Top 4 + Autre regroupé) |
| 15 | `CLV Segmentation Table HTML` | Clients | Tableau 4 quartiles (VIP / Premium / Standard / Occasionnel) |
| 16 | `Services par Categorie HTML` | Extras | 7 catégories de service avec ticket moyen |
| 17 | `Ratio Extras Cible HTML` | Extras | Jauge avec ligne cible 15 % et message de potentiel |

### 🔧 Pattern type d'une mesure HTML

Toutes les mesures HTML suivent le même squelette :

```dax
MaMesure HTML = 
VAR _data = ADDCOLUMNS( ALL(maTable[col]), "valeur", [Ma Mesure] )
VAR _max = MAXX(_data, [valeur])
VAR _rows = CONCATENATEX(_data,
    VAR _pct = FORMAT(ROUND([valeur] / _max * 100, 0), "0")
    VAR _color = SWITCH(TRUE(), [valeur] > seuil1, "#1FA67D", [valeur] > seuil2, "#E8C96A", "#E24B4A")
    RETURN "<div class='row'>...</div>",
    "",
    [valeur], DESC
)
RETURN "<style>...</style>" & _rows
```

Charte couleurs HotelChain dans les mesures HTML :
- **Or principal** : `#E8C96A` ou `#D4A631`
- **Vert leader** : `#1FA67D`
- **Rouge sous-perf** : `#E24B4A`
- **Orange intermédiaire** : `#E28B2D`
- **Blanc cassé** : `#F5E8D8`
- **Texte titre** : or sur fond noir

### 🔧 Installer le visuel HTML Content

1. Bandeau **Visualisations** → **...** → **Obtenir d'autres visuels**
2. Rechercher **"HTML Content"** par Daniel Marsh-Patrick
3. **Add** → glisser le visuel sur la page
4. **Champs → Values** → glisser la mesure HTML correspondante

## 4.9 Dossier `12. Segmentation Clients` (4 mesures)

```dax
Nb Clients Nouveaux = 
CALCULATE(
    DISTINCTCOUNT(hotelchain_clients_clv[client_id]),
    hotelchain_clients_clv[client_fidele] = 0
)
```

```dax
Nb Clients Fideles = 
CALCULATE(
    DISTINCTCOUNT(hotelchain_clients_clv[client_id]),
    hotelchain_clients_clv[client_fidele] = 1
)
```

```dax
% Clients Nouveaux = 
DIVIDE([Nb Clients Nouveaux], DISTINCTCOUNT(hotelchain_clients_clv[client_id]))
```
*Valeur attendue sans filtre : ~76,5 %.*

```dax
% Clients Fideles = 
DIVIDE([Nb Clients Fideles], DISTINCTCOUNT(hotelchain_clients_clv[client_id]))
```
*Valeur attendue sans filtre : ~23,5 %.*

## 4.10 Dossier `18. Recommandations Actions` (3 mesures)

Ces 3 mesures alimentent les 3 cards d'action de la page Extras (« 01 Conférences », « 02 Spa upselling », « 03 Cible 15 % »).

```dax
Reco Action 01 = 
VAR _ticket_conf = [Ticket Moyen Conference]
VAR _ticket_resto = CALCULATE(AVERAGE(services[montant]), services[categorie] = "Restaurant")
VAR _ratio = DIVIDE(_ticket_conf, _ticket_resto)
RETURN "Ticket " & FORMAT(_ticket_conf/1000, "0") & "k FCFA, " & FORMAT(_ratio, "0") & "x le restaurant. Priorite commerciale N1."
```

```dax
Reco Action 02 = 
VAR _ticket_spa = CALCULATE(AVERAGE(services[montant]), services[categorie] = "Spa")
RETURN FORMAT(_ticket_spa/1000, "0") & "k FCFA ticket moyen. Proposer systematiquement au check-in."
```

```dax
Reco Action 03 = 
VAR _ratio_actuel = [Ratio Extras vs Chambres]
VAR _potentiel = (0.15 - _ratio_actuel) * [Revenu Total]
RETURN "Atteindre la norme sectorielle = +" & FORMAT(_potentiel/1000000, "0") & " M FCFA/an additionnels."
```
*Calcule dynamiquement : si on atteint 15 % au lieu de 9,7 %, gain = (15 % - 9,7 %) × Revenu chambres ≈ +153 M FCFA/an.*

---
# V — Design system

## 5.1 Charte graphique HotelChain — Dark Luxury

HotelChain assume un parti-pris esthétique **noir + or**, qui évoque l'hôtellerie haut de gamme. Tout est sobre, contrasté, premium.

### Palette

| Rôle | Hex | Usage |
|---|---|---|
| Fond page | `#0F0F0F` ou `#1A1A1A` | Arrière-plan général |
| Or principal | `#E8C96A` / `#D4A631` | Titres, KPIs hero, accents, item actif sidebar |
| Or pâle | `#F5E8D8` | Variation, highlights |
| Vert success | `#1FA67D` | Leader RevPAR, CSAT > 4.0, variation positive |
| Orange | `#E28B2D` | ADR élevé, CSAT proche de la cible |
| Rouge | `#E24B4A` | Sous-performance, CSAT < 4.0, annulation |
| Blanc cassé | `#F5E8D8` | Texte sur fond noir |
| Gris séparateur | `#3A3A3A` | Bordures de cards, séparateurs |

### Typographie

| Élément | Police | Taille | Poids |
|---|---|---|---|
| Titre de page | Georgia / Playfair Display | 28 | 700 |
| Hero KPI | Playfair Display | 48 | 700 |
| Sous-titre dynamique | Segoe UI Light | 14 | 300 |
| Label KPI | Segoe UI | 11 | 600 (UPPERCASE) |
| Texte courant | Segoe UI | 13 | 400 |

## 5.2 Mockup PowerPoint → fonds PNG d'arrière-plan

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes (cards à ombre, sidebars). La méthode pro :
1. Dessiner la mise en page dans **PowerPoint** (mockup vierge sans données)
2. Exporter en **PNG haute résolution** (1280×720 ou 2001×1125)
3. Importer comme **arrière-plan de page** dans Power BI
4. Poser les visuels Power BI **par-dessus**

### Ce que le mockup PPTX vierge doit contenir

✅ **À inclure :**
- Sidebar de navigation noire avec onglets `01 Vue executive` → `05 Extras` (item actif en or)
- Logo HotelChain West (logo HW or sur fond noir) + DataProjectLab Academy en footer
- Cards / rectangles vides avec border-top accent or
- Encadrés vides pour les charts et tableaux

❌ **À NE PAS inclure :**
- Titre de page (zone de texte Power BI dynamique)
- Slicers Hôtel / Année (segments natifs)
- Toute valeur de KPI ou texte dans les cards
- Données dans les charts ou tableaux

### 🔧 Méthode 1 — Export PNG depuis PowerPoint (recommandé)

PowerPoint exporte par défaut en 96 DPI. Pour un 150 DPI lisible :

1. **Win + R** → `regedit` → **Entrée**
2. Aller dans `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
3. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)**
4. Nom : `ExportBitmapResolution` — Valeur : `150` (décimal)
5. Fermer regedit, **redémarrer PowerPoint**

Puis : **Fichier** → **Enregistrer sous** → **PNG** → **Toutes les diapositives**.

### 🔧 Méthode 2 — CloudConvert

1. Aller sur [cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png)
2. Charger `HotelChainWest_Mockup_V2_DarkLuxury_blank.pptx`
3. Options → 150 DPI → 1280×720
4. **Convert** → télécharger les 6 PNG

### Renommage final

```
bg-00-cover.png
bg-01-vue-executive.png
bg-02-occupation.png
bg-03-revenus.png
bg-04-clients.png
bg-05-extras.png
```

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page** (icône pinceau au niveau page)
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement de l'image** → **Adapter**
4. **Transparence** → **0 %**

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/03_page1.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VI — Construire les 6 pages

## 6.0 Page 0 — Couverture

> *Page d'accueil de présentation, accessible avant la navigation.*

| Zone | Visuel | Contenu |
|---|---|---|
| Titre principal | Zone de texte | « HOTEL CHAIN WEST » sur 2 lignes (Playfair 80pt or) |
| Sous-titre | Zone de texte | « ANALYTICS » + filet or + « Dashboard Power BI » |
| 5 KPIs cover | 5 Cartes | `Total Reservations` (8 000), `Revenu Total` formaté en Md FCFA, `CSAT Moyen` (4,01), `Taux Annulation` (3,1 %), `Total hotel` (5) |
| Sidebar nav (footer) | Boutons | 5 boutons textuels alignés bas : `01 Vue executive` `02 Occupation` `03 Revenus` `04 Clients` `05 Extras` |


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/04_page_cover.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.1 Page 1 — Vue executive

> *« Quelle est la santé globale de la chaîne ce trimestre ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Slicer Hôtel | Liste déroulante | `hotels[nom]` |
| Slicer Année | Liste horizontale | `Calendrier[Annee]` (2022, 2023, 2024) |
| KPI 1 | Carte (border-top or) | `Revenu Total` formaté Md FCFA |
| KPI 2 | Carte (border-top vert) | `CSAT Moyen` |
| KPI 3 | Carte (border-top orange) | `Taux Annulation` |
| KPI 4 | Carte (border-top blanc) | `Nuits Vendues` |
| RevPAR mensuel par hôtel | **HTML Content** | `Cards RevPAR Hotels` (5 cards verticales : Douala vert leader, autres en or, Accra rouge sous-perf.) |
| Revenu par canal | Barres horizontales | `hotelchain_canaux[canal]` × `hotelchain_canaux[revenu_total_fcfa]`, couleur par canal |
| Synthèse revenus | 3 barres | Revenu chambres (or), Revenu extras (vert), Total consolidé (blanc cassé) |


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/05_page_vue_executive.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.2 Page 2 — Occupation & Saisonnalité

> *« Quel hôtel performe, à quelle saison, avec quel pic / creux ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| KPI Pic | Carte (border-top vert) | `Pic Occupation` + `Pic Occupation Label` (ex: "Douala Janvier 2024 = 7,5 %") |
| KPI Creux | Carte (border-top rouge) | `Creux Occupation` + `Creux Occupation Label` (ex: "Accra Juin 2024 = 1,2 %") |
| KPI Moyenne | Carte (border-top or) | `Moyenne Occupation` + `Moyenne Occupation Label` (ex: "Tous les hôtels = 4,0 %") |
| Heatmap principale | **HTML Content** | `Heatmap Occupation HTML` (5 hôtels × 12 mois, palette dorée 5 paliers, valeurs % dans les cellules) |
| Titre heatmap | Carte | `Titre Heatmap Occupation` ("Taux d'occupation 2024 — hôtel × mois (%)") |
| Bilan par hôtel | 5 Cartes multi-rows | Pour chaque hôtel : `Hotel Label Etoiles` (titre) + `Hotel RevPAR FCFA` (valeur) + `Hotel Insight` (sous-titre coloré : Leader vert, 2e/3e rang or, ADR élevé orange, Sous-perf. rouge) |

### 📘 Comment lire la heatmap

Les colonnes sont les mois (J F M A M J J A S O N D), les lignes les hôtels (Douala, Cocody, Dakar, Plateau, Accra). Plus la cellule est dorée intense, plus l'hôtel est rempli ce mois-là. La saisonnalité ressort instantanément (pic janvier-mars, creux juin).


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/06_page_occupation.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.3 Page 3 — Revenus & Tarification

> *« À quel prix vendons-nous, quel hôtel a le meilleur sweet spot ADR × Occupation ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| ADR catalogue par type chambre | **HTML Content** | `ADR par Type Chambre HTML` (Suite 403k, Deluxe 209k, Supérieure 139k, Standard 91k — barres or/vert/orange/blanc + nb réservations) |
| Sweet spot ADR × Taux Occupation | Nuage de points | X : `ADR`, Y : `Taux Occupation`, taille : `Revenu Total`, légende : `hotels[nom]`, points colorés (Accra rouge, Plateau gris, Dakar orange, Cocody jaune-vert, Douala vert) |
| RevPAR par hôtel toute période | 5 Cartes | Pour chaque hôtel : `Hotel Label Etoiles` (entête) + `Hotel RevPAR FCFA` (valeur en gros) avec barre de progression colorée selon rang (Douala 8 151, Dakar 6 239, Accra 4 851, Cocody 6 440, Plateau 5 287) |

### 📘 Le sweet spot ADR × Occupation

L'idéal d'un hôtel = en haut à droite (ADR élevé ET occupation élevée). En bas à gauche = double échec (prix bas et hôtel vide). Sur HotelChain :
- **Douala** : sweet spot (ADR moyen, occupation élevée)
- **Accra** : double pénalité (ADR bas et occupation très basse) → revoir le positionnement
- **Dakar / Plateau** : ADR élevé mais occupation moyenne → tester des promotions ciblées pour remplir


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/07_page_revenus.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.4 Page 4 — Clients & Satisfaction

> *« Qui sont nos clients, quelle est leur satisfaction, quelle est la valeur des fidèles ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| CSAT moyen par hôtel | **HTML Content** | `CSAT par Hotel HTML` (5 barres horizontales avec ligne cible pointillée or à 4.0 ; vert si ≥ 4, orange proche, rouge si < 4) |
| Segmentation clients CLV | **HTML Content** | `CLV Segmentation Table HTML` (4 quartiles : Q4 VIP or 51 % du revenu, Q3 Premium vert 27 %, Q2 Standard orange 16 %, Q1 Occasionnel rouge 7 %) |
| Top 5 nationalités | **HTML Content** | `Top Nationalites HTML` (Ivoirien 30 % or, Sénégalais 15 % vert, Camerounais 12,8 % blanc cassé, Ghanéen 10,1 % orange, Autre 32,2 % bleu) |
| Fidèles vs Nouveaux | 2 Cartes hero | `% Clients Nouveaux` (76,4 % or) et `% Clients Fideles` (23,6 % vert) |

### 📘 La logique des quartiles inversés

Dans la table CLV, le quartile Q1 contient les clients **avec le plus haut revenu** (top 25 %). Pour l'affichage, on inverse l'étiquette : **Q1 dans la donnée = Q4 VIP affiché**. Cohérent avec la sémantique "Q4 = haut" attendue par les utilisateurs métier.


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/08_page_clients.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.5 Page 5 — Services & Extras (Revenus additionnels)

> *« Combien rapportent les services annexes, où est le potentiel d'upsell ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Ratio extras / revenu chambre (jauge) | **HTML Content** | `Ratio Extras Cible HTML` (jauge or remplie à 9,7 %, ligne cible pointillée à 15 %, sous-titre "Potentiel +153 M FCFA/an en atteignant 15 %") |
| KPI 1 | Carte (border-top or) | `Revenu Extras` (278 M FCFA Extras Total) |
| KPI 2 | Carte (border-top vert) | `Ticket Moyen Conference` (132 504 FCFA) |
| KPI 3 | Carte (border-top orange) | `Ratio Extras vs Chambres` (9,7 %) |
| Revenus extras par catégorie | **HTML Content** | `Services par Categorie HTML` (Restaurant 68,9M or, Salle conférence 65,1M vert, Spa 52,6M blanc cassé, Room service 33,5M orange, Transport 29,1M orange, Minibar 19,5M bleu, Blanchisserie 9,4M bleu — chaque ligne avec ticket moyen en sous-titre) |
| 3 cards d'action | 3 Cartes multi-rows | Card 01 : « Conférences » + `Reco Action 01` (border vert) ; Card 02 : « Spa upselling » + `Reco Action 02` (border or) ; Card 03 : « Cible 15 % » + `Reco Action 03` (border or pâle) |

### 📘 Le potentiel +153 M FCFA

Calcul : si on passe de 9,7 % à 15 % (norme sectorielle), gain = (15 % − 9,7 %) × Revenu chambres ≈ 5,3 % × 2,87 Md FCFA ≈ **+153 M FCFA/an**. C'est le chiffre qui rend l'upsell tangible — il ne s'agit pas d'un objectif moral mais d'une opportunité chiffrée.


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/09_page_extras.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VII — Slicers, navigation, finitions

**Slicers globaux** sur chaque page : `hotels[nom]` (liste déroulante or), `Calendrier[Annee]` (3 boutons 2022 / 2023 / 2024 — bouton actif rempli or). Clic droit sur chaque slicer → **Synchroniser les segments → cocher toutes les pages**.

**Navigation sidebar verticale** sur chaque page : 5 textes cliquables `01 Vue executive` → `05 Extras`. Item actif en or `#E8C96A`, items inactifs en gris `#888`. Implémenter via boutons natifs avec **action Navigation de page**.

**Logo HotelChain West (HW)** : carré noir avec lettres HW dorées en haut de la sidebar — image PNG à insérer en haut de chaque page.

**Footer** : « DataProjectLab Academy » en gris discret en bas de la sidebar.

---
# VIII — Validation et livraison

## 8.1 Checklist de recette

**Modèle** : 9 tables sources + Calendrier + _Mesures, 9 relations actives, 0 LocalDateTable, relation services↔reservations en inactive (volontaire) · `Calendrier` marquée comme table de dates · Auto Date/Time désactivé.

**Mesures** : 44 dans `_Mesures` · 18 dossiers numérotés `01.` à `18.` · format défini (FCFA, %, nombre) · descriptions remplies sur toutes les mesures (visible dans l'éditeur).

**Visuels HTML Content** : 9 visuels marketplace actifs (Heatmap, RevPAR Cards, ADR, CSAT, Top Nat, CLV, Services cat, Ratio Extras, Breakdown).

**Pages** : 6 pages (Cover + 5 nav) · Slicers Hôtel/Année synchronisés · Sidebar avec navigation par boutons · Couleurs conformes à la charte Dark Luxury.

**Performance** : ouverture < 5 s · aucun visuel en erreur.

## 8.2 Pièges fréquents et solutions

| Symptôme | Cause | Correction |
|---|---|---|
| `Taux Annulation` affiche une date au lieu d'un % | Power Query a inféré `taux_annulation_pct` en DateTime | Forcer Decimal Number en Power Query, ou hack `DAY([col]) + MONTH([col])/10` |
| `PREVIOUSMONTH` renvoie blank | `Calendrier` non marquée comme table de dates | Outils de table → Marquer comme table de dates |
| Visuel HTML Content vide | Marketplace pas installé ou mesure ne retourne pas une string | Installer HTML Content de Daniel Marsh-Patrick + tester la mesure dans une carte |
| Heatmap : palette ne ressort pas | Couleurs trop pâles vs fond noir | Utiliser des paliers d'or saturés (#E8C96A → #B8860B) avec contraste élevé |
| `Taux Occupation` faux (60 % au lieu de 53 %) | Moyenne de ratios | Utiliser `DIVIDE(SUM(nuits), SUM(capacite))` — pas `AVERAGE(taux_occupation)` |
| Sweet spot scatter : tous les points superposés | Échelle automatique cassée | Format → Axe X → Min/Max manuels (ex: 50k-450k FCFA) |
| Cards d'hôtel n'affichent rien | Mesure `Hotel Insight` retourne blank pour les rangs intermédiaires | La mesure renvoie `""` (string vide) volontairement — afficher quand même la valeur RevPAR |
| `CLV Segmentation Table HTML` affiche les quartiles dans le mauvais ordre | Quartile data Q1 = top, mais utilisateur attend Q4 = top | L'inversion est faite dans la mesure HTML — vérifier la logique SWITCH |

## 8.3 Storytelling exécutif

Pour présenter au directeur général, suis l'ordre des 5 pages :

1. **Vue executive** : « 2024 boucle à 2,87 Md FCFA, CSAT 4,01, taux d'annulation 3,1 %. Booking.com et Direct sont à parité (2,4k vs 2,3k réservations). »
2. **Occupation** : « Pic à Douala janvier 2024 (7,5 %), creux à Accra juin (1,2 %). La chaîne est en sous-régime structurel — moyenne 4 %. Forte saisonnalité hiver/été. »
3. **Revenus** : « Douala est notre leader (RevPAR 8 151 FCFA), Accra à la traîne (4 851 FCFA). La Suite vendue à 403k FCFA fonctionne bien (728 nuits) — capitaliser. »
4. **Clients** : « 76 % de nouveaux clients = belle acquisition mais faible rétention. Les VIP (Q4 = 25 % des clients) génèrent 51 % du revenu. Programme fidélité urgent. »
5. **Extras** : « Ratio extras / chambres à 9,7 % — sous le benchmark 15 %. **+153 M FCFA/an** disponibles si on atteint la cible. Conférences = ticket 132k = priorité commerciale n°1. »

## 8.4 Annexes

### Règles DAX universelles à graver

1. Toujours `DIVIDE` (jamais `/`)
2. Pour les ratios pondérés (taux d'occupation), `DIVIDE(SUM, SUM)` au lieu de `AVERAGE(ratio)`
3. Pattern HTML Content : `CONCATENATEX` + style inline + `RETURN "<style>...</style>" & _rows`
4. `SUBSTITUTE([col], "prefixe ", "")` pour nettoyer un préfixe (ex: "HotelChain Douala" → "Douala")
5. `RANKX(ALL(col), [Mesure], , DESC)` pour le rang descendant
6. Descriptions de mesure remplies : visible dans l'éditeur, sert de doc vivante

### Mapping mockup PPTX ↔ pages Power BI

| Slide PPTX | Background PNG | Page Power BI |
|---|---|---|
| 0 | `bg-00-cover.png` | Couverture |
| 1 | `bg-01-vue-executive.png` | Vue executive |
| 2 | `bg-02-occupation.png` | Occupation & Saisonnalité |
| 3 | `bg-03-revenus.png` | Revenus & Tarification |
| 4 | `bg-04-clients.png` | Clients & Satisfaction |
| 5 | `bg-05-extras.png` | Services & Extras |


---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">HotelChain West — Hospitality Analytics</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>